In [1]:
# the parameters from MCMC results to form the parameters file for simulations

import pandas as pd
import pprint

def read_nml_file(file_path):
    """ read nml file content """
    with open(file_path, "r") as file:
        lines = file.readlines()
    return lines

def parse_nml_lines(lines):
    """ parse the lines in a nml-format with multiple groups """
    data = {}
    current_group = None
    for line in lines:
        line = line.strip()
        if line.startswith('&'): # start a new group
            current_group = line[1:]
            data[current_group] = {}
        elif line.startswith('/'): # end current group
            current_group = None
        elif "=" in line and current_group:
            key, value = line.split('=', 1)
            data[current_group][key.strip()] = value.strip()
    return data

def update_nml_data(data, group, updates):
    """ update the nml date for the specified group, where the updates is a dictionary containing the key-value pairs to be updated  """
    if group in data:
        for key, value in updates.items():
            if key in data[group]:
                data[group][key] = value

def write_nml_file(file_path, data):
    """ re-write the updated nml data back to the file, keeping the group structure """
    with open(file_path, 'w') as file:
        for group, items in data.items():
            file.write(f'&{group}\n')
            for key, value in items.items():
                file.write(f'{key} = {value}\n')
            file.write('/\n')


def run_params4nml(file_params, org_data, file_path):
    # read the parameter file 
    df_dat = pd.read_csv(file_params).iloc[-1]
    print("test len: ", len(df_dat))
    print(df_dat)

    df_dict = {}
    for idx in df_dat.index:
        if idx.split("_")[-1].rstrip() in ["Tree", "Shrub", "Sphagnum"]:
            item,_,_ = idx.rpartition('_')
            if item == "SLA": item = "SLAx"
            if item in df_dict.keys():
                df_dict[item] = df_dict[item] + ", " + str(df_dat[idx])
            else:
                df_dict[item] = str(df_dat[idx])
        else:
            df_dict[idx] = str(df_dat[idx])
    print(df_dict)
    # update

    nml_name = ["nml_site_params", "nml_species_params"]
    # group_to_update = 'nml_species_params'  # 
    for group_to_update in nml_name:
        for ikey, item in df_dict.items():
            updates = {ikey.strip(): item}  # the content for updating
            update_nml_data(org_data, group_to_update, updates)

    write_nml_file(file_path, org_data)

def get_init_data(file_simu, dict_old, file_path):
    df_simu_data = pd.read_csv(file_simu, sep=",")
    # print(df_simu_data)
    dict_res_site = {}
    dict_res_site["cLit_m"]    = df_simu_data["cLitter"].values[-1]
    dict_res_site["cLit_s"]    = df_simu_data["cLitterCwd"].values[-1]
    dict_res_site["cSoil_f"]   = df_simu_data["cSoilFast"].values[-1]
    dict_res_site["cSoil_s"]   = df_simu_data["cSoilSlow"].values[-1]
    dict_res_site["cSoil_p"]   = df_simu_data["cSoilPassive"].values[-1]
    dict_res_site["QNminer"]   = df_simu_data["nMineral"].values[-1]
    dict_res_site["liq_water"] = str(df_simu_data["mrso_1"].values[-1]/1000) + "," + str(df_simu_data["mrso_2"].values[-1]/1000) + "," + \
                                 str(df_simu_data["mrso_3"].values[-1]/1000) + "," + str(df_simu_data["mrso_4"].values[-1]/1000) + "," + \
                                 str(df_simu_data["mrso_5"].values[-1]/1000) + "," + str(df_simu_data["mrso_6"].values[-1]/1000) + "," + \
                                 str(df_simu_data["mrso_7"].values[-1]/1000) + "," + str(df_simu_data["mrso_8"].values[-1]/1000) + "," + \
                                 str(df_simu_data["mrso_9"].values[-1]/1000) + "," + str(df_simu_data["mrso_10"].values[-1]/1000)
    dict_res_site["zwt"]       = df_simu_data["wtd"].values[-1]*1000
    dict_res_site["CH4"]       = str(df_simu_data["CH4_1"].values[-1]) + "," + str(df_simu_data["CH4_2"].values[-1]) + "," + \
                                 str(df_simu_data["CH4_3"].values[-1]) + "," + str(df_simu_data["CH4_4"].values[-1]) + "," + \
                                 str(df_simu_data["CH4_5"].values[-1]) + "," + str(df_simu_data["CH4_6"].values[-1]) + "," + \
                                 str(df_simu_data["CH4_7"].values[-1]) + "," + str(df_simu_data["CH4_8"].values[-1]) + "," + \
                                 str(df_simu_data["CH4_9"].values[-1]) + "," + str(df_simu_data["CH4_10"].values[-1])
    dict_res_sps = {}
    dict_res_sps["cLeaf"] = str(df_simu_data["cLeaf_Tree"].values[-1]) + "," + str(df_simu_data["cLeaf_Shrub"].values[-1]) + "," + str(df_simu_data["cLeaf_Sphagnum"].values[-1])
    dict_res_sps["cStem"] = str(df_simu_data["cStem_Tree"].values[-1]) + "," + str(df_simu_data["cStem_Shrub"].values[-1]) + "," + str(df_simu_data["cStem_Sphagnum"].values[-1])
    dict_res_sps["cRoot"] = str(df_simu_data["cRoot_Tree"].values[-1]) + "," + str(df_simu_data["cRoot_Shrub"].values[-1]) + "," + str(df_simu_data["cRoot_Sphagnum"].values[-1])
    dict_res_sps["nsc"]   = str(df_simu_data["nsc_Tree"].values[-1])   + "," + str(df_simu_data["nsc_Shrub"].values[-1])   + "," + str(df_simu_data["nsc_Sphagnum"].values[-1])
    dict_res_sps["nsn"]   = str(df_simu_data["nsn_Tree"].values[-1])   + "," + str(df_simu_data["nsn_Shrub"].values[-1])   + "," + str(df_simu_data["nsn_Sphagnum"].values[-1])
    print(dict_res_site, dict_res_sps)
    nml_name = ["nml_site_initial_values", "nml_species_initial_values"]
    for group_to_update in nml_name:
        if group_to_update == "nml_site_initial_values": dict_to_save = dict_res_site
        else: dict_to_save = dict_res_sps
        for ikey, item in dict_to_save.items():
            # updates = {ikey.strip(): item}  # the content for updating
            # update_nml_data(org_data, group_to_update, updates)
            dict_old[group_to_update][ikey] = item
    print("data:",dict_old["nml_site_initial_values"])
    write_nml_file(file_path, dict_old)




ls_plots = ["P04", "P06", "P08", "P10", "P11", "P13", "P16", "P17", "P19", "P20"]

org_file = "parameters.nml" # destination file for nml-format
# ls_plots = ["P04"]

# each plot
for idx_plot, iplot in enumerate(ls_plots):
    print(iplot)
    file_simu = f"../../outputs/run_mcmc_pretreat_{iplot}/results_csv_format/TECO-SPRUCE_run_mcmc_pretreat_{iplot}_Hourly.csv"
    file_path = f"../inputs/in_treat/{iplot}/parameters.nml"#f"inputs/in_pretreat/{iplot}/parameters.nml"
    lines = read_nml_file(file_path)
    data  = parse_nml_lines(lines)
    # run_params4nml(file_params, data, file_path)
    # print(data["nml_site_initial_values"]) # nml_species_initial_values
    pprint.pprint(data["nml_site_initial_values"])
    res_simu = get_init_data(file_simu, data, file_path)

P04
{'CH4': '0.0,0.0055,0.014,0.019,0.1151,3.8045,3.8216,3.7695,3.7428,3.7352',
 'CH4_V': '0., 0., 0., 0., 0., 0., 0., 0., 0., 0.',
 'CN0_lit_m': '40.',
 'CN0_lit_s': '300.',
 'CN0_soil_f': '10.',
 'CN0_soil_p': '12.',
 'CN0_soil_s': '20.',
 'FRLEN': '0.75, 0.2, 0.02, 0.02, 0.01, 0.0, 0.0, 0.0, 0.0, 0.0',
 'G': '20.5',
 'N_deposit': '2.34',
 'Nbub': '100.',
 'QNminer': '1.4492',
 'Tice': '0.0',
 'Tsnow': '-20.',
 'Tsoill': '-0.09, 0.73, 1.3, 1.95, 2.3, 3., 4., 4.5, 5., 5.98,',
 'Twater': '0.0',
 'Vp': '0., 0., 0., 0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01',
 'albedo_snow': '0.7',
 'b_bound': '100.',
 'bubble_methane_tot': '0.',
 'cLit_m': '473.0652',
 'cLit_s': '488.6241',
 'cSoil_f': '157.3168',
 'cSoil_p': '46138.9859',
 'cSoil_s': '40283.2743',
 'condu_b': '0.08',
 'condu_snow': '0.1',
 'dcount': '50.',
 'dcount_soil': '50.',
 'decay_m': '2.2192',
 'depth_1': '10.0',
 'depth_ex': '0.05',
 'diff_s': '1.',
 'diff_snow': '1.8',
 'fa': '1',
 'fsub': '0.1',
 'fwsoil': '1.0',
 'ice': '0.02